# 5. Data Cleaning


> **Dataset note:** This project uses a synthetic OTT viewer-behaviour dataset for educational analysis.  
> **Hotstar is used only as the business case/scenario; the data is not claimed to be proprietary Hotstar data.**

**Allowed tools:** Python, NumPy, Pandas, Matplotlib, Seaborn.  
**Not used:** Scikit-learn, Plotly, Power BI, Tableau, machine learning, or statistical libraries beyond NumPy/Pandas.

## Process Performed
I preserved the raw data, standardized labels/text, checked missing values and duplicates, validated numeric ranges and binary columns, verified release years, and only applied safe type correction where justified.

In [1]:
# PROCESS: Load the CSV into a Pandas DataFrame and verify that the file loaded correctly.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

DATA_PATH = "ott_viewer_dropoff_retention_us_v1.0.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (33171, 23)


,show_id,title,platform,genre,release_year,season_number,episode_number,episode_duration_min,pacing_score,hook_strength,dialogue_density,visual_intensity,avg_watch_percentage,pause_count,rewind_count,skip_intro,cognitive_load,attention_required,night_watch_safe,drop_off,drop_off_probability,retention_risk,dataset_version
0,66732,Stranger Things,Netflix,Sci-Fi & Fantasy,2016.0,1,1,48,4,5,high,5,39,3,0,0,9,high,0,1,0.649,high,v1.0
1,66732,Stranger Things,Netflix,Sci-Fi & Fantasy,2016.0,1,2,55,5,4,low,8,55,3,3,1,5,medium,0,0,0.473,medium,v1.0
2,66732,Stranger Things,Netflix,Sci-Fi & Fantasy,2016.0,1,3,51,4,8,high,7,46,4,2,0,9,high,0,0,0.583,medium,v1.0
3,66732,Stranger Things,Netflix,Sci-Fi & Fantasy,2016.0,1,4,50,4,7,medium,3,50,4,1,0,7,high,0,0,0.520,medium,v1.0
4,66732,Stranger Things,Netflix,Sci-Fi & Fantasy,2016.0,1,5,52,4,3,low,4,35,3,0,1,7,high,0,1,0.638,high,v1.0


## Cleaning Strategy

This notebook checks quality first and applies conservative cleaning only when justified. It does **not** silently alter the analytical meaning of the dataset.

In [2]:
# KERNEL-SAFE SETUP
# Keep plotting memory controlled while preserving the analysis.
import gc

MAX_PLOT_ROWS = 5000

def safe_sample(data, n=MAX_PLOT_ROWS, random_state=42):
    """Use the full data when small; otherwise use a reproducible sample for plotting."""
    return data if len(data) <= n else data.sample(n=n, random_state=random_state)

def finish_plot():
    """Render and release the current Matplotlib figure."""
    plt.tight_layout()
    plt.show()
    plt.close()
    gc.collect()

In [3]:
# PROCESS: Create a clean working copy and standardize the dataset safely.

# Create an independent working copy so the raw DataFrame remains available.
clean_df = df.copy()

# ---------------------------------------------------------
# STEP 1: Standardize column names to snake_case
# ---------------------------------------------------------
# Rules:
# - convert labels to strings
# - remove leading/trailing spaces
# - convert to lowercase
# - replace spaces/special characters with underscores
# - collapse repeated underscores
# - remove leading/trailing underscores

import re

def clean_column_name(column_name):
    """Convert one column label to clean snake_case."""
    name = str(column_name).strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_")


clean_df.columns = [
    clean_column_name(col)
    for col in clean_df.columns
]

print("Standardized columns:")
print(clean_df.columns.tolist())


# ---------------------------------------------------------
# STEP 2: Identify text / categorical columns safely
# ---------------------------------------------------------
# is_string_dtype() is used for text detection so the code works with
# newer Pandas string dtypes without Pandas4Warning.

categorical_columns = [
    col
    for col in clean_df.columns
    if pd.api.types.is_string_dtype(clean_df[col])
]


# ---------------------------------------------------------
# STEP 3: Clean text values
# ---------------------------------------------------------
# Remove unnecessary leading and trailing spaces.

for col in categorical_columns:
    clean_df[col] = clean_df[col].str.strip()


# ---------------------------------------------------------
# STEP 4: Identify numerical columns
# ---------------------------------------------------------

numerical_columns = (
    clean_df
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)


# ---------------------------------------------------------
# STEP 5: Name the row index clearly
# ---------------------------------------------------------

clean_df.index.name = "row_id"


# ---------------------------------------------------------
# STEP 6: Display a concise cleaning summary
# ---------------------------------------------------------

print("\nNumerical columns:")
print(numerical_columns)

print("\nCategorical/text columns:")
print(categorical_columns)

print("\nNumber of numerical columns:", len(numerical_columns))
print("Number of categorical/text columns:", len(categorical_columns))

print("\nCleaned shape:", clean_df.shape)
print("Row index name:", clean_df.index.name)

display(clean_df.head())

Standardized columns:
['show_id', 'title', 'platform', 'genre', 'release_year', 'season_number', 'episode_number', 'episode_duration_min', 'pacing_score', 'hook_strength', 'dialogue_density', 'visual_intensity', 'avg_watch_percentage', 'pause_count', 'rewind_count', 'skip_intro', 'cognitive_load', 'attention_required', 'night_watch_safe', 'drop_off', 'drop_off_probability', 'retention_risk', 'dataset_version']

Numerical columns:
['show_id', 'release_year', 'season_number', 'episode_number', 'episode_duration_min', 'pacing_score', 'hook_strength', 'visual_intensity', 'avg_watch_percentage', 'pause_count', 'rewind_count', 'skip_intro', 'cognitive_load', 'night_watch_safe', 'drop_off', 'drop_off_probability']

Categorical/text columns:
['title', 'platform', 'genre', 'dialogue_density', 'attention_required', 'retention_risk', 'dataset_version']

Number of numerical columns: 16
Number of categorical/text columns: 7

Cleaned shape: (33171, 23)
Row index name: row_id


,show_id,title,platform,genre,release_year,season_number,episode_number,episode_duration_min,pacing_score,hook_strength,dialogue_density,visual_intensity,avg_watch_percentage,pause_count,rewind_count,skip_intro,cognitive_load,attention_required,night_watch_safe,drop_off,drop_off_probability,retention_risk,dataset_version
row_id,,,,,,,,,,,,,,,,,,,,,,,
0,66732,Stranger Things,Netflix,Sci-Fi & Fantasy,2016.0,1,1,48,4,5,high,5,39,3,0,0,9,high,0,1,0.649,high,v1.0
1,66732,Stranger Things,Netflix,Sci-Fi & Fantasy,2016.0,1,2,55,5,4,low,8,55,3,3,1,5,medium,0,0,0.473,medium,v1.0
2,66732,Stranger Things,Netflix,Sci-Fi & Fantasy,2016.0,1,3,51,4,8,high,7,46,4,2,0,9,high,0,0,0.583,medium,v1.0
3,66732,Stranger Things,Netflix,Sci-Fi & Fantasy,2016.0,1,4,50,4,7,medium,3,50,4,1,0,7,high,0,0,0.520,medium,v1.0
4,66732,Stranger Things,Netflix,Sci-Fi & Fantasy,2016.0,1,5,52,4,3,low,4,35,3,0,1,7,high,0,1,0.638,high,v1.0


## Missing-Value Audit

In [4]:
# PROCESS: Audit missing values before deciding whether any treatment is required.
missing_before = clean_df.isna().sum()
display(missing_before[missing_before > 0].sort_values(ascending=False))

# We do not automatically impute missing values because that can distort EDA.
# Instead, later analyses use dropna() only on the columns required by that analysis.

Series([], dtype: int64)

## Duplicate Audit

In [5]:
# PROCESS: Check for exact duplicate records so repeated rows do not bias the analysis.
duplicates_before = clean_df.duplicated().sum()
print("Exact duplicate rows:", duplicates_before)

if duplicates_before > 0:
    clean_df = clean_df.drop_duplicates().copy()

print("Shape after exact-duplicate removal:", clean_df.shape)

Exact duplicate rows: 0
Shape after exact-duplicate removal: (33171, 23)


## Range and Category Validation

In [6]:
# PROCESS: Count category frequencies and percentages to understand how records are distributed.
numeric_checks = {
    "avg_watch_percentage": (0, 100),
    "episode_duration_min": (0, np.inf),
    "pause_count": (0, np.inf),
    "rewind_count": (0, np.inf),
    "drop_off_probability": (0, 1),
}

for col, (low, high) in numeric_checks.items():
    if col in clean_df.columns:
        invalid = (~clean_df[col].between(low, high, inclusive="both")) & clean_df[col].notna()
        print(f"{col}: {invalid.sum()} out-of-range values")

for col in ["skip_intro", "night_watch_safe", "drop_off"]:
    if col in clean_df.columns:
        print(f"\n{col} values:")
        print(clean_df[col].value_counts(dropna=False))

avg_watch_percentage: 0 out-of-range values
episode_duration_min: 0 out-of-range values
pause_count: 0 out-of-range values
rewind_count: 0 out-of-range values
drop_off_probability: 0 out-of-range values

skip_intro values:
skip_intro
0    16809
1    16362
Name: count, dtype: int64

night_watch_safe values:
night_watch_safe
0    32644
1      527
Name: count, dtype: int64

drop_off values:
drop_off
0    28367
1     4804
Name: count, dtype: int64


## Additional Data-Quality Validation

The dataset currently contains **no missing values and no exact duplicate rows**, so no imputation or duplicate deletion is necessary.  
I still validate year integrity, score ranges, binary indicators, identifier fields, and empty text values so the notebook demonstrates the complete cleaning process rather than assuming the data is clean.

In [7]:
# PROCESS: Perform additional integrity checks and apply only a safe release-year type correction.

# 1) Confirm whether release_year can be represented as an integer.
#    The source CSV stores it as float, but every observed value is a whole year.
year_is_integer_like = (
    clean_df["release_year"].notna().all()
    and np.all(clean_df["release_year"] % 1 == 0)
)

print("release_year contains only whole years:", year_is_integer_like)

if year_is_integer_like:
    clean_df["release_year"] = clean_df["release_year"].astype(int)
    print("release_year safely converted to:", clean_df["release_year"].dtype)

# 2) Validate score fields that are expected to be on a 1–10 scale.
score_columns = [
    "pacing_score", "hook_strength",
    "visual_intensity", "cognitive_load"
]

for col in score_columns:
    invalid_count = (~clean_df[col].between(1, 10)).sum()
    print(f"{col}: {invalid_count} values outside the expected 1–10 range")

# 3) Validate binary 0/1 indicators.
binary_columns = ["skip_intro", "night_watch_safe", "drop_off"]

for col in binary_columns:
    invalid_count = (~clean_df[col].isin([0, 1])).sum()
    print(f"{col}: {invalid_count} values outside {{0, 1}}")

# 4) Validate season and episode numbering.
print("Invalid season numbers (< 1):", (clean_df["season_number"] < 1).sum())
print("Invalid episode numbers (< 1):", (clean_df["episode_number"] < 1).sum())

# 5) Check essential text columns for blank strings.
essential_text = ["title", "platform", "genre", "retention_risk"]
for col in essential_text:
    blank_count = clean_df[col].fillna("").str.strip().eq("").sum()
    print(f"{col}: {blank_count} blank values")

release_year contains only whole years: True
release_year safely converted to: int64
pacing_score: 0 values outside the expected 1–10 range
hook_strength: 0 values outside the expected 1–10 range
visual_intensity: 0 values outside the expected 1–10 range
cognitive_load: 0 values outside the expected 1–10 range
skip_intro: 0 values outside {0, 1}
night_watch_safe: 0 values outside {0, 1}
drop_off: 0 values outside {0, 1}
Invalid season numbers (< 1): 0
Invalid episode numbers (< 1): 0
title: 0 blank values
platform: 0 blank values
genre: 0 blank values
retention_risk: 0 blank values


## Final Cleaned Dataset

In [8]:
# PROCESS: Perform this analysis step and inspect the resulting table/output before drawing conclusions.
df = clean_df.copy()
print("Final shape:", df.shape)
df.head()

Final shape: (33171, 23)


,show_id,title,platform,genre,release_year,season_number,episode_number,episode_duration_min,pacing_score,hook_strength,dialogue_density,visual_intensity,avg_watch_percentage,pause_count,rewind_count,skip_intro,cognitive_load,attention_required,night_watch_safe,drop_off,drop_off_probability,retention_risk,dataset_version
row_id,,,,,,,,,,,,,,,,,,,,,,,
0,66732,Stranger Things,Netflix,Sci-Fi & Fantasy,2016,1,1,48,4,5,high,5,39,3,0,0,9,high,0,1,0.649,high,v1.0
1,66732,Stranger Things,Netflix,Sci-Fi & Fantasy,2016,1,2,55,5,4,low,8,55,3,3,1,5,medium,0,0,0.473,medium,v1.0
2,66732,Stranger Things,Netflix,Sci-Fi & Fantasy,2016,1,3,51,4,8,high,7,46,4,2,0,9,high,0,0,0.583,medium,v1.0
3,66732,Stranger Things,Netflix,Sci-Fi & Fantasy,2016,1,4,50,4,7,medium,3,50,4,1,0,7,high,0,0,0.520,medium,v1.0
4,66732,Stranger Things,Netflix,Sci-Fi & Fantasy,2016,1,5,52,4,3,low,4,35,3,0,1,7,high,0,1,0.638,high,v1.0


### Interpretation Note
The outputs above describe patterns in the supplied **synthetic episode-level dataset**. They should support business hypotheses and content investigation, not be presented as causal proof or proprietary Hotstar findings.

## Reusable Validation Function

Instead of writing separate range-check logic repeatedly, this function performs the same validation consistently across numerical fields.

In [9]:
def check_range(data, column, minimum, maximum):
    """Return rows whose values fall outside the expected range."""
    invalid = data[
        (data[column] < minimum) |
        (data[column] > maximum)
    ]

    print(
        f"{column}: {len(invalid)} invalid values "
        f"outside [{minimum}, {maximum}]"
    )

    return invalid

# Validate the most important numeric fields.
check_range(clean_df, "avg_watch_percentage", 0, 100)
check_range(clean_df, "drop_off_probability", 0, 1)
check_range(clean_df, "hook_strength", 1, 10)
check_range(clean_df, "pacing_score", 1, 10)
check_range(clean_df, "cognitive_load", 1, 10)

avg_watch_percentage: 0 invalid values outside [0, 100]
drop_off_probability: 0 invalid values outside [0, 1]
hook_strength: 0 invalid values outside [1, 10]
pacing_score: 0 invalid values outside [1, 10]
cognitive_load: 0 invalid values outside [1, 10]


,show_id,title,platform,genre,release_year,season_number,episode_number,episode_duration_min,pacing_score,hook_strength,dialogue_density,visual_intensity,avg_watch_percentage,pause_count,rewind_count,skip_intro,cognitive_load,attention_required,night_watch_safe,drop_off,drop_off_probability,retention_risk,dataset_version
row_id,,,,,,,,,,,,,,,,,,,,,,,


## Episode-Level Duplicate Check

Exact-row duplicates and duplicate episode identities are different problems.  
This check tests whether the same show-season-episode combination appears more than once.

In [10]:
episode_key_cols = ["show_id", "season_number", "episode_number"]

duplicate_episode_records = clean_df[
    clean_df.duplicated(
        subset=episode_key_cols,
        keep=False
    )
].sort_values(episode_key_cols)

print("Rows involved in duplicate episode keys:",
      len(duplicate_episode_records))

if len(duplicate_episode_records) > 0:
    display(duplicate_episode_records.head(20))

Rows involved in duplicate episode keys: 2568


,show_id,title,platform,genre,release_year,season_number,episode_number,episode_duration_min,pacing_score,hook_strength,dialogue_density,visual_intensity,avg_watch_percentage,pause_count,rewind_count,skip_intro,cognitive_load,attention_required,night_watch_safe,drop_off,drop_off_probability,retention_risk,dataset_version
row_id,,,,,,,,,,,,,,,,,,,,,,,
1676,1620,CSI: Miami,Paramount Plus Apple TV Channel,Drama,2002,1,1,45,5,6,low,3,58,3,0,0,5,medium,0,0,0.429,medium,v1.0
1700,1620,CSI: Miami,Paramount Plus Apple TV Channel,Drama,2002,1,1,45,3,10,high,3,53,7,0,0,9,high,0,0,0.551,medium,v1.0
1677,1620,CSI: Miami,Paramount Plus Apple TV Channel,Drama,2002,1,2,44,4,4,low,9,39,4,4,1,7,high,0,1,0.615,high,v1.0
1701,1620,CSI: Miami,Paramount Plus Apple TV Channel,Drama,2002,1,2,44,4,7,high,7,36,1,6,0,9,high,0,1,0.613,high,v1.0
1678,1620,CSI: Miami,Paramount Plus Apple TV Channel,Drama,2002,1,3,45,5,7,high,7,59,1,1,0,7,high,0,0,0.450,medium,v1.0
1702,1620,CSI: Miami,Paramount Plus Apple TV Channel,Drama,2002,1,3,45,4,6,medium,5,44,5,0,0,7,high,0,0,0.572,medium,v1.0
1679,1620,CSI: Miami,Paramount Plus Apple TV Channel,Drama,2002,1,4,43,4,7,low,4,49,2,1,0,7,high,0,0,0.504,medium,v1.0
1703,1620,CSI: Miami,Paramount Plus Apple TV Channel,Drama,2002,1,4,43,6,3,low,6,55,3,5,1,5,medium,0,0,0.487,medium,v1.0
1680,1620,CSI: Miami,Paramount Plus Apple TV Channel,Drama,2002,1,5,43,3,5,low,8,48,6,6,1,7,high,0,0,0.579,medium,v1.0


### Important Naming Decision

The project keeps the original analytical field names such as:

- `avg_watch_percentage`
- `episode_duration_min`
- `drop_off_probability`
- `retention_risk`

These names are already clear, valid `snake_case`, and match the project brief.  
They are **not renamed unnecessarily**, because changing them would make the notebook harder to compare with the supplied documentation.

### Naming Decision

The cleaned dataset keeps field names such as `avg_watch_percentage`,
`episode_duration_min`, `drop_off_probability`, and `retention_risk`
because they already follow valid Python `snake_case` conventions and
match the supplied project documentation.